# Main3 Cross-Model OOD Analysis

This notebook reads the rebuttal cross-model Main3 transfer outputs and summarizes:

- Table-3-style family summaries with **models** as the OOD axis
- selected AUROC and PR-AUC transfer matrices
- calibration curves for the winning family/target panels
- false-positive rates at fixed recall levels
- top features for the selected models


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
RESULTS_DIR = Path('/playpen-ssd/smerrill/deception2/rebuttal/results/OOD_Modeling_main3_cross_model_ood_xgb_pca_128')
assert RESULTS_DIR.exists(), f'Missing results dir: {RESULTS_DIR}'

CONFIG_PATH = RESULTS_DIR / 'config.csv'
INVENTORY_PATH = RESULTS_DIR / 'bundle_inventory.csv'
SPLIT_SUMMARY_PATH = RESULTS_DIR / 'split_summary.csv'
FEATURE_SPACE_PATH = RESULTS_DIR / 'feature_space_catalog.csv'
TRANSFER_PATH = RESULTS_DIR / 'all_transfer_metrics.csv'
SUMMARY_PATH = RESULTS_DIR / 'transfer_summary.csv'
TRAIN_MODEL_PATH = RESULTS_DIR / 'train_model_summary.csv'
PANEL_PATH = RESULTS_DIR / 'best_feature_space_by_target_size_family.csv'
BEST_MODEL_PATH = RESULTS_DIR / 'best_model_by_target_size_family.csv'
CALIBRATION_PATH = RESULTS_DIR / 'all_calibration_curves.csv'
FPR_PATH = RESULTS_DIR / 'all_fpr_at_recall.csv'
TOP_FEATURES_PATH = RESULTS_DIR / 'top_features_for_best_models.csv'
MANIFEST_PATH = RESULTS_DIR / 'selected_family_panel_tables' / 'panel_table_manifest.csv'

config_df = pd.read_csv(CONFIG_PATH) if CONFIG_PATH.exists() else pd.DataFrame()
inventory_df = pd.read_csv(INVENTORY_PATH) if INVENTORY_PATH.exists() else pd.DataFrame()
split_summary_df = pd.read_csv(SPLIT_SUMMARY_PATH) if SPLIT_SUMMARY_PATH.exists() else pd.DataFrame()
feature_space_df = pd.read_csv(FEATURE_SPACE_PATH) if FEATURE_SPACE_PATH.exists() else pd.DataFrame()
metrics_df = pd.read_csv(TRANSFER_PATH) if TRANSFER_PATH.exists() else pd.DataFrame()
summary_df = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else pd.DataFrame()
train_model_df = pd.read_csv(TRAIN_MODEL_PATH) if TRAIN_MODEL_PATH.exists() else pd.DataFrame()
panel_df = pd.read_csv(PANEL_PATH) if PANEL_PATH.exists() else pd.DataFrame()
best_model_df = pd.read_csv(BEST_MODEL_PATH) if BEST_MODEL_PATH.exists() else pd.DataFrame()
calibration_df = pd.read_csv(CALIBRATION_PATH) if CALIBRATION_PATH.exists() else pd.DataFrame()
fpr_df = pd.read_csv(FPR_PATH) if FPR_PATH.exists() else pd.DataFrame()
top_features_df = pd.read_csv(TOP_FEATURES_PATH) if TOP_FEATURES_PATH.exists() else pd.DataFrame()
manifest_df = pd.read_csv(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()

print('RESULTS_DIR:', RESULTS_DIR)
print('transfer rows:', len(metrics_df))
print('summary rows:', len(summary_df))
print('calibration rows:', len(calibration_df))
print('fpr rows:', len(fpr_df))


## Run Config

In [ ]:
config_df

## Inventory

In [ ]:
inventory_df

## Split Summary

In [ ]:
split_summary_df

In [ ]:
FEATURE_LABELS = {
    'tfidf_baseline': 'TF-IDF baseline',
    'attention_only': 'Attention only',
    'activation_only': 'Activation only: PCA final',
    'attention_plus_activation': 'Attention + PCA final',
    'baseline_raw': 'Raw final activation',
}
TARGET_LABELS = {
    'delta_pos_gt_0_3': 'Deceptive commitment',
    'delta_neg_lt_neg_0_3': 'Honest commitment',
}
FAMILY_ORDER = ['tfidf_baseline', 'attention_only', 'activation_only', 'attention_plus_activation', 'baseline_raw']

def family_label(value):
    return FEATURE_LABELS.get(str(value), str(value))

def target_label(value):
    return TARGET_LABELS.get(str(value), str(value))


## Table 3 Style Summary

In [ ]:
REQUESTED_FEATURE_SIZE = 128
if panel_df.empty:
    print('No panel summary found.')
else:
    table_df = panel_df.loc[panel_df['requested_feature_size'].eq(REQUESTED_FEATURE_SIZE)].copy()
    table_df['family_label'] = table_df['feature_family_group'].map(family_label)
    table_df['target_label'] = table_df['target_name'].map(target_label)
    table_df['family_sort'] = table_df['feature_family_group'].map({name: idx for idx, name in enumerate(FAMILY_ORDER)})
    auroc_table = (
        table_df.sort_values(['family_sort', 'target_label'])
        .pivot(index='family_label', columns='target_label', values='mean_ood_auroc')
    )
    prauc_table = (
        table_df.sort_values(['family_sort', 'target_label'])
        .pivot(index='family_label', columns='target_label', values='mean_ood_pr_auc')
    )
    brier_table = (
        table_df.sort_values(['family_sort', 'target_label'])
        .pivot(index='family_label', columns='target_label', values='mean_ood_brier')
    )
    print('AUROC')
    display(auroc_table.style.format('{:.3f}'))
    print('PR-AUC')
    display(prauc_table.style.format('{:.3f}'))
    print('Brier')
    display(brier_table.style.format('{:.3f}'))


## Winning Feature Spaces

In [ ]:
if panel_df.empty:
    print('No panel summary found.')
else:
    cols = [
        'target_name', 'requested_feature_size', 'feature_family_group',
        'selected_feature_space_title', 'selected_feature_count',
        'mean_ood_auroc', 'mean_ood_pr_auc', 'mean_ood_brier', 'alignment_detail'
    ]
    panel_df.loc[:, cols].sort_values(['requested_feature_size', 'target_name', 'feature_family_group'])


## AUROC Transfer Matrices

In [ ]:
if manifest_df.empty:
    print('No panel manifest found.')
else:
    requested_feature_size = 128
    subset = manifest_df.loc[manifest_df['requested_feature_size'].eq(requested_feature_size)].copy()
    for _, row in subset.iterrows():
        matrix_path = Path(row['auroc_matrix_path'])
        if not matrix_path.exists():
            continue
        matrix_df = pd.read_csv(matrix_path, index_col=0)
        plt.figure(figsize=(5, 4))
        sns.heatmap(matrix_df, annot=True, fmt='.3f', cmap='magma', vmin=0.0, vmax=1.0)
        plt.title(f"AUROC | {target_label(row['target_name'])} | {family_label(row['feature_family_group'])}")
        plt.xlabel('Eval model')
        plt.ylabel('Train model')
        plt.tight_layout()
        plt.show()


## PR-AUC Transfer Matrices

In [ ]:
if manifest_df.empty:
    print('No panel manifest found.')
else:
    requested_feature_size = 128
    subset = manifest_df.loc[manifest_df['requested_feature_size'].eq(requested_feature_size)].copy()
    for _, row in subset.iterrows():
        matrix_path = Path(row['pr_auc_matrix_path'])
        if not matrix_path.exists():
            continue
        matrix_df = pd.read_csv(matrix_path, index_col=0)
        plt.figure(figsize=(5, 4))
        sns.heatmap(matrix_df, annot=True, fmt='.3f', cmap='viridis', vmin=0.0, vmax=1.0)
        plt.title(f"PR-AUC | {target_label(row['target_name'])} | {family_label(row['feature_family_group'])}")
        plt.xlabel('Eval model')
        plt.ylabel('Train model')
        plt.tight_layout()
        plt.show()


## Calibration Curves

In [ ]:
if calibration_df.empty or panel_df.empty:
    print('Calibration or panel data missing.')
else:
    requested_feature_size = 128
    winners = panel_df.loc[panel_df['requested_feature_size'].eq(requested_feature_size)].copy()
    for _, win in winners.iterrows():
        subset = calibration_df.loc[
            calibration_df['target_name'].eq(win['target_name'])
            & calibration_df['feature_space'].eq(win['selected_feature_space'])
            & calibration_df['feature_size_label'].eq(win['source_feature_size_label'])
        ].copy()
        if subset.empty:
            continue
        plt.figure(figsize=(5, 4))
        plt.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
        plotted = False
        for eval_role, role_df in subset.groupby('eval_role'):
            mean_curve = role_df.groupby('bin_idx', as_index=False).agg(mean_pred=('mean_pred', 'mean'), frac_pos=('frac_pos', 'mean'))
            if mean_curve.empty:
                continue
            plotted = True
            plt.plot(mean_curve['mean_pred'], mean_curve['frac_pos'], marker='o', label=eval_role)
        if plotted:
            plt.title(f"Calibration | {target_label(win['target_name'])} | {family_label(win['feature_family_group'])}")
            plt.xlabel('Mean predicted probability')
            plt.ylabel('Observed positive rate')
            plt.legend(loc='best')
            plt.tight_layout()
            plt.show()
        else:
            plt.close()


## False-Positive Rate at Fixed Recall

In [ ]:
if fpr_df.empty or panel_df.empty:
    print('FPR or panel data missing.')
else:
    requested_feature_size = 128
    winners = panel_df.loc[panel_df['requested_feature_size'].eq(requested_feature_size)].copy()
    rows = []
    for _, win in winners.iterrows():
        subset = fpr_df.loc[
            fpr_df['target_name'].eq(win['target_name'])
            & fpr_df['feature_space'].eq(win['selected_feature_space'])
            & fpr_df['feature_size_label'].eq(win['source_feature_size_label'])
            & fpr_df['eval_role'].eq('ood')
        ].copy()
        if subset.empty:
            continue
        summary = subset.groupby('recall_target', as_index=False).agg(mean_fpr=('fpr', 'mean'))
        summary['target_label'] = target_label(win['target_name'])
        summary['family_label'] = family_label(win['feature_family_group'])
        rows.append(summary)
    if rows:
        summary_df = pd.concat(rows, ignore_index=True)
        display(summary_df.pivot(index=['family_label', 'target_label'], columns='recall_target', values='mean_fpr').style.format('{:.3f}'))
    else:
        print('No selected-panel FPR rows found.')


## Top Features

In [ ]:
if top_features_df.empty:
    print('No top-feature exports found.')
else:
    top_features_df.head(50)
